In [5]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, classification_report
from imblearn.under_sampling import RandomUnderSampler


In [6]:
TW_500= pd.read_csv(
    "../classification/Twitter/Relative_labeling/sigma=500/Twitter-Relative-Sigma-500.data",
    sep=",",
    header=None
)

TW_1000= pd.read_csv(
    "../classification/Twitter/Relative_labeling/sigma=1000/Twitter-Relative-Sigma-1000.data",
    sep=",",
    header=None
)
TW_1500= pd.read_csv(
    "../classification/Twitter/Relative_labeling/sigma=1500/Twitter-Relative-Sigma-1500.data",
    sep=",",
    header=None
)
groups = ["NCD", 'AI', 'AS(NA)', 'BL',
         'NAC', 'AS(NAC)', 'CS', 'AT', 'NA','ADL', 'NAD']

columns = []
for group in groups:
    for t in range(7):
        columns.append(f"{group}_{t}")

columns.append("label") 

TW_500.columns = columns
TW_1000.columns = columns
TW_1500.columns = columns

TW_500.head(2)

,NCD_0,NCD_1,NCD_2,NCD_3,NCD_4,NCD_5,NCD_6,AI_0,AI_1,AI_2,...,ADL_5,ADL_6,NAD_0,NAD_1,NAD_2,NAD_3,NAD_4,NAD_5,NAD_6,label
0,889,939,960,805,805,1143,1121,549,613,587,...,1.0,1.0,889,939,960,805,805,1143,1121,0.0
1,542,473,504,626,647,795,832,366,288,318,...,1.0,1.0,542,473,504,626,647,795,832,1.0


### RandomUnderSampler for Random Forest

In [7]:
X = TW_500.drop(columns=['label'])
y = TW_500['label']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [8]:

X = TW_500.drop(columns=['label'])
y = TW_500['label']

# --- 2. split ---
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# --- 3. ratios to test ---
ratios = {
    "1:10": 0.1,
    "1:15": 0.067,
    "1:20": 0.05,
    "1:25": 0.04,
    "1:30": 0.033
}

results = {}

for name, ratio in ratios.items():
    print(f"\n===== Undersampling {name} =====")

    # undersampling
    rus = RandomUnderSampler(sampling_strategy=ratio, random_state=42)
    X_res, y_res = rus.fit_resample(X_train, y_train)

    print("After:", y_res.value_counts())

    # model
    rf = RandomForestClassifier(random_state=42)

    param_grid = {
        'n_estimators': [100],
        'max_depth': [20, None],
        'min_samples_split': [2, 5],
        'min_samples_leaf': [1, 2]
    }

    grid = GridSearchCV(
        rf,
        param_grid,
        scoring='f1',
        cv=3,
        n_jobs=-1
    )

    grid.fit(X_res, y_res)

    best_model = grid.best_estimator_

    # prediction
    y_pred = best_model.predict(X_test)

    f1 = f1_score(y_test, y_pred)
    results[name] = f1

    print("Best params:", grid.best_params_)
    print("F1:", f1)

# --- 4. summary ---
print("\n===== FINAL RESULTS =====")
for k, v in results.items():
    print(f"{k}: {v:.4f}")


===== Undersampling 1:10 =====
After: label
0.0    28960
1.0     2896
Name: count, dtype: int64
Best params: {'max_depth': None, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 100}
F1: 0.5849387040280211

===== Undersampling 1:15 =====
After: label
0.0    43223
1.0     2896
Name: count, dtype: int64
Best params: {'max_depth': None, 'min_samples_leaf': 2, 'min_samples_split': 5, 'n_estimators': 100}
F1: 0.6059379217273954

===== Undersampling 1:20 =====
After: label
0.0    57920
1.0     2896
Name: count, dtype: int64
Best params: {'max_depth': 20, 'min_samples_leaf': 2, 'min_samples_split': 5, 'n_estimators': 100}
F1: 0.6230483271375464

===== Undersampling 1:25 =====
After: label
0.0    72400
1.0     2896
Name: count, dtype: int64
Best params: {'max_depth': 20, 'min_samples_leaf': 2, 'min_samples_split': 5, 'n_estimators': 100}
F1: 0.6290196078431373

===== Undersampling 1:30 =====
After: label
0.0    87757
1.0     2896
Name: count, dtype: int64
Best params: {'max_dept

### RandomUnderSampler for XGBoost

In [9]:
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import f1_score
from imblearn.under_sampling import RandomUnderSampler
from xgboost import XGBClassifier

ratios = {
    "1:10": 0.1,
    "1:15": 0.067,
    "1:20": 0.05,
    "1:25": 0.04,
    "1:30": 0.033
}

results = {}

for name, ratio in ratios.items():
    print(f"\n===== Undersampling {name} =====")

    # undersampling
    rus = RandomUnderSampler(sampling_strategy=ratio, random_state=42)
    X_res, y_res = rus.fit_resample(X_train, y_train)

    print("After:", y_res.value_counts())

    # model
    xgb = XGBClassifier(
        random_state=42,
        eval_metric='logloss'
    )

    # grid
    param_grid = {
        'n_estimators': [100],
        'max_depth': [4, 6],
        'learning_rate': [0.05, 0.1]
    }

    grid = GridSearchCV(
        xgb,
        param_grid,
        scoring='f1',
        cv=3,
        n_jobs=-1
    )

    grid.fit(X_res, y_res)

    best_model = grid.best_estimator_
    y_pred = best_model.predict(X_test)

    f1 = f1_score(y_test, y_pred)
    results[name] = f1

    print("Best params:", grid.best_params_)
    print("F1:", f1)

# --- фінальна таблиця ---
print("\n===== FINAL RESULTS =====")
for k, v in results.items():
    print(f"{k}: {v:.4f}")


===== Undersampling 1:10 =====
After: label
0.0    28960
1.0     2896
Name: count, dtype: int64
Best params: {'learning_rate': 0.05, 'max_depth': 6, 'n_estimators': 100}
F1: 0.5754385964912281

===== Undersampling 1:15 =====
After: label
0.0    43223
1.0     2896
Name: count, dtype: int64
Best params: {'learning_rate': 0.05, 'max_depth': 6, 'n_estimators': 100}
F1: 0.5899665551839465

===== Undersampling 1:20 =====
After: label
0.0    57920
1.0     2896
Name: count, dtype: int64
Best params: {'learning_rate': 0.1, 'max_depth': 6, 'n_estimators': 100}
F1: 0.626453488372093

===== Undersampling 1:25 =====
After: label
0.0    72400
1.0     2896
Name: count, dtype: int64
Best params: {'learning_rate': 0.1, 'max_depth': 4, 'n_estimators': 100}
F1: 0.6133743274404304

===== Undersampling 1:30 =====
After: label
0.0    87757
1.0     2896
Name: count, dtype: int64
Best params: {'learning_rate': 0.1, 'max_depth': 6, 'n_estimators': 100}
F1: 0.595879556259905

===== FINAL RESULTS =====
1:10: 0.

### Tomek Links  Random Forest

In [10]:
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, classification_report
from imblearn.under_sampling import TomekLinks

# --- 1. data ---
X = TW_500.drop(columns=['label'])
y = TW_500['label']

# --- 2. split ---
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# --- 3. Tomek Links ---
tl = TomekLinks()
X_train_clean, y_train_clean = tl.fit_resample(X_train, y_train)

print("Before:", y_train.value_counts())
print("After:", y_train_clean.value_counts())

# --- 4. model ---
rf = RandomForestClassifier(random_state=42)

# --- 5. grid ---
param_grid = {
    'n_estimators': [100],
    'max_depth': [20, None],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2],
    'class_weight': [None, 'balanced']
}

# --- 6. grid search ---
grid = GridSearchCV(
    rf,
    param_grid,
    scoring='f1',
    cv=3,
    n_jobs=-1
)

grid.fit(X_train_clean, y_train_clean)

# --- 7. evaluation ---
best_model = grid.best_estimator_
y_pred = best_model.predict(X_test)

print("Best params:", grid.best_params_)
print("F1:", f1_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Before: label
0.0    109669
1.0      2896
Name: count, dtype: int64
After: label
0.0    109136
1.0      2896
Name: count, dtype: int64
Best params: {'class_weight': None, 'max_depth': None, 'min_samples_leaf': 2, 'min_samples_split': 2, 'n_estimators': 100}
F1: 0.61082910321489
              precision    recall  f1-score   support

         0.0       0.99      1.00      0.99     27418
         1.0       0.79      0.50      0.61       724

    accuracy                           0.98     28142
   macro avg       0.89      0.75      0.80     28142
weighted avg       0.98      0.98      0.98     28142



### Tomek Links  XGBoost

In [11]:
from imblearn.under_sampling import TomekLinks
from xgboost import XGBClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import f1_score, classification_report

# --- Tomek Links ---
tl = TomekLinks()
X_train_clean, y_train_clean = tl.fit_resample(X_train, y_train)

print("Before:", y_train.value_counts())
print("After:", y_train_clean.value_counts())

# --- model ---
xgb = XGBClassifier(
    random_state=42,
    eval_metric='logloss'
)

# --- grid ---
param_grid = {
    'n_estimators': [100],
    'max_depth': [4, 6],
    'learning_rate': [0.05, 0.1],
    'subsample': [0.8, 1],
    'colsample_bytree': [0.8, 1]
}

# --- grid search ---
grid = GridSearchCV(
    xgb,
    param_grid,
    scoring='f1',
    cv=3,
    n_jobs=-1
)

grid.fit(X_train_clean, y_train_clean)

# --- evaluation ---
best_model = grid.best_estimator_
y_pred = best_model.predict(X_test)

print("Best params:", grid.best_params_)
print("F1:", f1_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Before: label
0.0    109669
1.0      2896
Name: count, dtype: int64
After: label
0.0    109136
1.0      2896
Name: count, dtype: int64
Best params: {'colsample_bytree': 0.8, 'learning_rate': 0.1, 'max_depth': 6, 'n_estimators': 100, 'subsample': 1}
F1: 0.6113328012769353
              precision    recall  f1-score   support

         0.0       0.99      0.99      0.99     27418
         1.0       0.72      0.53      0.61       724

    accuracy                           0.98     28142
   macro avg       0.86      0.76      0.80     28142
weighted avg       0.98      0.98      0.98     28142



### Tomek Links Random Forest

In [12]:
tl = TomekLinks()
X_train_clean, y_train_clean = tl.fit_resample(X_train, y_train)

print("Before:", y_train.value_counts())
print("After:", y_train_clean.value_counts())

rf = RandomForestClassifier(random_state=42)

param_grid = {
    'n_estimators': [100],
    'max_depth': [20, None],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2],
    'class_weight': [None, 'balanced']
}

grid = GridSearchCV(
    rf,
    param_grid,
    scoring='f1',
    cv=3,
    n_jobs=-1
)

grid.fit(X_train_clean, y_train_clean)

best_model = grid.best_estimator_
y_pred = best_model.predict(X_test)

print("Best params:", grid.best_params_)
print("F1:", f1_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Before: label
0.0    109669
1.0      2896
Name: count, dtype: int64
After: label
0.0    109136
1.0      2896
Name: count, dtype: int64
Best params: {'class_weight': None, 'max_depth': None, 'min_samples_leaf': 2, 'min_samples_split': 2, 'n_estimators': 100}
F1: 0.61082910321489
              precision    recall  f1-score   support

         0.0       0.99      1.00      0.99     27418
         1.0       0.79      0.50      0.61       724

    accuracy                           0.98     28142
   macro avg       0.89      0.75      0.80     28142
weighted avg       0.98      0.98      0.98     28142



### ADASYN XGBoost

In [13]:
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import f1_score, classification_report
from imblearn.over_sampling import ADASYN
from xgboost import XGBClassifier

# --- data ---
X = TW_500.drop(columns=['label'])
y = TW_500['label']

# --- split ---
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# --- ADASYN ---
adasyn = ADASYN(sampling_strategy=0.1, random_state=42)
X_res, y_res = adasyn.fit_resample(X_train, y_train)

print("After:", y_res.value_counts())

# --- model ---
xgb = XGBClassifier(
    random_state=42,
    eval_metric='logloss'
)

# --- grid ---
param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [4, 6],
    'learning_rate': [0.05, 0.1]
}

# --- grid search ---
grid = GridSearchCV(
    xgb,
    param_grid,
    scoring='f1',
    cv=3,
    n_jobs=-1
)

grid.fit(X_res, y_res)

# --- evaluation ---
best_model = grid.best_estimator_
y_pred = best_model.predict(X_test)

print("Best params:", grid.best_params_)
print("F1:", f1_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

After: label
0.0    109669
1.0     11187
Name: count, dtype: int64
Best params: {'learning_rate': 0.1, 'max_depth': 6, 'n_estimators': 200}
F1: 0.5951283739302172
              precision    recall  f1-score   support

         0.0       0.99      0.99      0.99     27418
         1.0       0.57      0.62      0.60       724

    accuracy                           0.98     28142
   macro avg       0.78      0.81      0.79     28142
weighted avg       0.98      0.98      0.98     28142



### ADASYN XGBoost Random Forest

In [14]:
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import f1_score, classification_report
from imblearn.over_sampling import ADASYN
from sklearn.ensemble import RandomForestClassifier

# --- data ---
X = TW_500.drop(columns=['label'])
y = TW_500['label']

# --- split ---
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# --- ADASYN ---
adasyn = ADASYN(sampling_strategy=0.1, random_state=42)
X_res, y_res = adasyn.fit_resample(X_train, y_train)

print("After:", y_res.value_counts())

# --- model ---
rf = RandomForestClassifier(
    random_state=42
)

# --- grid ---
param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2]
}

# --- grid search ---
grid = GridSearchCV(
    rf,
    param_grid,
    scoring='f1',
    cv=3,
    n_jobs=-1
)

grid.fit(X_res, y_res)

# --- evaluation ---
best_model = grid.best_estimator_
y_pred = best_model.predict(X_test)

print("Best params:", grid.best_params_)
print("F1:", f1_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

After: label
0.0    109669
1.0     11187
Name: count, dtype: int64
Best params: {'max_depth': None, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 200}
F1: 0.6086330935251798
              precision    recall  f1-score   support

         0.0       0.99      0.99      0.99     27418
         1.0       0.64      0.58      0.61       724

    accuracy                           0.98     28142
   macro avg       0.81      0.79      0.80     28142
weighted avg       0.98      0.98      0.98     28142



### RUSBOOST 

In [15]:
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import f1_score, classification_report
from imblearn.ensemble import RUSBoostClassifier

X = TW_500.drop(columns=['label'])
y = TW_500['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# --- model ---
rusboost = RUSBoostClassifier(random_state=42)

# --- grid ---
param_grid = {
    'n_estimators': [50, 100, 200],
    'learning_rate': [0.05, 0.1, 0.2]
}

# --- grid search ---
grid = GridSearchCV(
    rusboost,
    param_grid,
    scoring='f1',
    cv=3,
    n_jobs=-1,
    verbose=1
)

grid.fit(X_train, y_train)

# --- evaluation ---
best_model = grid.best_estimator_
y_pred = best_model.predict(X_test)

print("Best params:", grid.best_params_)
print("F1:", f1_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Fitting 3 folds for each of 9 candidates, totalling 27 fits
Best params: {'learning_rate': 0.2, 'n_estimators': 50}
F1: 0.3028450505525511
              precision    recall  f1-score   support

         0.0       1.00      0.89      0.94     27418
         1.0       0.18      0.89      0.30       724

    accuracy                           0.89     28142
   macro avg       0.59      0.89      0.62     28142
weighted avg       0.98      0.89      0.93     28142

